In [ ]:
import pandas as pd
from tqdm import tqdm
from locations import (
    extract_geonames_coordinates,
    extract_wikidata_coordinates,
    query_geocoder,
)

In [ ]:
import os

if os.path.exists("locations.xlsx"):
    locations = pd.read_excel("locations.xlsx")
    locations = locations.fillna("")
else:
    locations = pd.DataFrame(columns=["location", "lat", "long", "label", "www.geonames.org", "www.wikidata.org", "www.giessen.de"])
locations

In [ ]:
# locations["lat", "long"] = locations.apply(
#     lambda row: (
#         extract_geonames_coordicates(row["www.geonames.org"])
#         if row["www.geonames.org"].strip()
#         else (
#             extract_wikidata_coordinates(row["www.wikidata.org"])
#             if row["www.wikidata.org"].strip()
#             else None
#         )
#     )
# )

for idx, row in tqdm(locations.iterrows(), total=locations.shape[0]):
    if row["lat"] and str(row["lat"]).strip():
        continue
    if row["www.geonames.org"].strip():
        coords = extract_geonames_coordinates(row["www.geonames.org"])
    elif row["www.wikidata.org"].strip():
        coords = extract_wikidata_coordinates(row["www.wikidata.org"])
    else:
        coords = query_geocoder(row["location"])

    locations.at[idx, "lat"] = coords["lat"] if coords else None
    locations.at[idx, "long"] = coords["long"] if coords else None
locations = locations.fillna("")
locations

In [ ]:
cols = list(locations.columns)
cols = [cols[0], cols[-2], cols[-1]] + cols[1:-2]
locations = locations[cols]

locations = locations.sort_values("location")
locations

In [ ]:
# Merge with existing file: only add new rows and fill empty cells
import os
import re

def _normalize_loc(name):
    """Normalize for matching: lowercase, no punctuation, sorted words."""
    name = str(name).strip().lower()
    name = re.sub(r'[^\w\s]', '', name)
    return ' '.join(sorted(name.split()))

if os.path.exists("locations.xlsx"):
    existing = pd.read_excel("locations.xlsx")
    existing = existing.fillna("")
    existing_idx = {_normalize_loc(n): i for i, n in enumerate(existing["location"])}
    # Add new rows and fill empty cells
    for _, row in locations.iterrows():
        key = _normalize_loc(row["location"])
        if key not in existing_idx:
            existing = pd.concat([existing, row.to_frame().T], ignore_index=True)
            existing_idx[key] = len(existing) - 1
        else:
            # Fill empty cells only
            idx = existing_idx[key]
            for col in locations.columns:
                existing_val = str(existing.at[idx, col]).strip()
                if existing_val in ("", "nan"):
                    new_val = str(row[col]).strip()
                    if new_val not in ("", "nan"):
                        existing.at[idx, col] = row[col]
    existing.to_excel("locations.xlsx", index=False)
else:
    locations.to_excel("locations.xlsx", index=False)